# Vision-LSTM Patch Traversal Experiments

In [ ]:
# ── Cell 1: Mount Drive, clone repo, install deps ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

!git clone https://github.com/ae-aydin/vision-lstm.git /content/vision-lstm 2>/dev/null \
    || git -C /content/vision-lstm pull

%cd /content/vision-lstm
sys.path.insert(0, '/content/vision-lstm')

# Colab already has torch, torchvision, matplotlib, tqdm — only einops is missing
!pip install einops==0.8.2 -q
print('Setup complete.')

In [ ]:
# ── Cell 2: Download and prepare Tiny ImageNet ─────────────────────────────
# Dataset goes to local Colab SSD (/content), NOT Google Drive.
from pathlib import Path

DATA_DIR = '/content/tiny-imagenet-200'

assert not DATA_DIR.startswith('/content/drive'), \
    'DATA_DIR must be on local SSD, not Google Drive!'

if not Path(DATA_DIR).exists():
    print('Downloading Tiny ImageNet...')
    !wget -q http://cs231n.stanford.edu/tiny-imagenet-200.zip -O /tmp/tiny-imagenet-200.zip
    !unzip -q /tmp/tiny-imagenet-200.zip -d /content/
    !rm /tmp/tiny-imagenet-200.zip
    print('Download complete.')

# Removes unused test/ split and restructures val into ImageFolder format.
# Safe to rerun — skips if already done.
!python scripts/prepare_tinyimagenet.py --data-dir {DATA_DIR}

In [ ]:
# ── Cell 3: Find max batch size and scale LR ───────────────────────────────
# Measurement runs with bf16 AMP active (same conditions as training).
import subprocess, re

result = subprocess.run(
    ['python', 'scripts/find_batch_size.py'],
    capture_output=True, text=True
)
print(result.stdout)

batch_match = re.search(r'Suggested batch\s*:\s*(\d+)', result.stdout)
lr_match    = re.search(r'Scaled LR\s*:\s*([\d.e+\-]+)', result.stdout)

BATCH_SIZE = int(batch_match.group(1)) if batch_match else 512
LR         = float(lr_match.group(1))  if lr_match    else 5e-4
EXP_DIR    = '/content/drive/MyDrive/vil-experiments'  # results saved to Drive
EPOCHS     = 30
SEED       = 42
COMPILE    = True   # torch.compile: faster training, first epoch ~5-10 min for tracing

print(f'Batch size : {BATCH_SIZE}')
print(f'LR         : {LR:.2e}')
print(f'Epochs     : {EPOCHS}')
print(f'Seed       : {SEED}')
print(f'Compile    : {COMPILE}')

In [ ]:
# ── Cell 4: Run experiments ─────────────────────────────────────────────────
# Comment out traversals already completed to avoid reruns.
compile_flag = '--compile' if COMPILE else ''

for traversal in ['rowwise', 'zigzag', 'spiral', 'hilbert', 'random']:
    print(f"\n{'='*60}\nTraversal: {traversal}\n{'='*60}")
    !python train.py \
        --traversal  {traversal} \
        --data-dir   {DATA_DIR} \
        --exp-dir    {EXP_DIR} \
        --batch-size {BATCH_SIZE} \
        --lr         {LR} \
        --epochs     {EPOCHS} \
        --seed       {SEED} \
        --workers    2 \
        {compile_flag}

print('\nAll experiments complete.')